In [2]:
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
from networkx.algorithms.tree import branching_weight

In [3]:
filtered_vocab_df = pd.read_json("../data/filtered_vocab_df.json")
filtered_vocab_df.head(10)

,word,1501-1550,1551-1600,1601-1650,1651-1700,Medicine,Astronomy/Astrology/Cosmography,Biology,Mathematics,Meteorology/Earth sciences,...,Biology_freq,Mathematics_freq,Meteorology/Earth sciences_freq,Physics_freq,Geography/Cartography_freq,Alchemy/Chemistry_freq,EMLAP_freq,mean_freq,shared,transl
2313,dico,43255,122549,42457,72224,107036,81298,76957,52414,44323,...,0.003069,0.002645,0.002988,0.002912,0.002816,0.002063,0.004034,0.002921,True,"say, call, tell"
4254,liber,39559,65900,38496,53100,97802,65756,65230,49280,50603,...,0.002602,0.002487,0.003411,0.003108,0.003290,0.004743,0.001314,0.002680,True,"book, volume, inner bark of a tree, book, book"
10443,possum,25029,82700,38391,76841,81305,74924,50255,50826,41844,...,0.002004,0.002565,0.002821,0.003553,0.001975,0.002350,0.004110,0.002563,True,"be able, can"
6100,facio,31814,98524,36944,51438,96530,62963,42272,45704,32760,...,0.001686,0.002307,0.002209,0.002487,0.001690,0.001891,0.004597,0.002367,True,"do, make, handle"
6099,habeo,31690,89199,39443,59770,81643,63141,61273,48571,33185,...,0.002444,0.002451,0.002237,0.002355,0.002320,0.001734,0.002807,0.002310,True,"have, hold, possess, consider, think"
1764,pars,33225,93059,44337,52385,95410,65986,42118,53361,31827,...,0.001680,0.002693,0.002146,0.002476,0.001962,0.001895,0.002397,0.002279,True,part
8953,res,29069,70669,28291,51565,74400,62681,42267,36572,40633,...,0.001686,0.001846,0.002739,0.002701,0.002085,0.002239,0.002556,0.002129,True,suddenly
1449,uideo,21536,59099,26372,50503,61322,53447,47212,28471,28146,...,0.001883,0.001437,0.001897,0.002244,0.001466,0.001554,0.001730,0.001717,True,
2323,locus,23102,56782,31421,43680,50814,57201,40768,31263,25466,...,0.001626,0.001578,0.001717,0.001703,0.002230,0.001240,0.001090,0.001632,True,"place, location"
56,aqua,11747,41231,20765,34038,42778,18714,30884,14948,28253,...,0.001232,0.000754,0.001905,0.001437,0.001160,0.001966,0.005758,0.001575,True,"get/fetch/bring water, be watered"


In [4]:
with open("../data/vectors_dict_comp.pkl", "rb") as file:
    vectors_dict = pickle.load(file)

To validate our models, we have turned to a synonym benchmark dataset developed by Sprugnoli et al. and avaialble from GitHub: https://github.com/CIRCSE/Lemma-Embeddings-for-Latin/blob/master/syn-selection-benchmark-Latin.tsv

In [5]:
# Python
import pandas as pd

cols = ["target", "related", "unrelated1", "unrelated2", "unrelated3"]

benchmark = pd.read_csv(
    "https://raw.githubusercontent.com/CIRCSE/Lemma-Embeddings-for-Latin/refs/heads/master/syn-selection-benchmark-Latin.tsv",
    sep="\t",
    header=None,       # first row stays as data
    names=cols         # set your column names
)

In [36]:
benchmark = benchmark.replace(" ", "")

In [37]:
vectors_dict["LASLA"].similarity("aqua", "sal")

0.5070169

In [105]:
# Python
import math
CANDS = ["related", "unrelated1", "unrelated2", "unrelated3"]

def get_sims(vectors_dict, subcorpus, row, fallback=math.nan):
    kv = vectors_dict[subcorpus]
    t = row["target"]
    out = {}
    out["target_present"] = t in kv
    for c in CANDS:
        w = row[c]
        present = (w in kv) and out["target_present"]
        out[f"sim_{c}"] = kv.similarity(t, w) if present else fallback
    return out

# Build the DataFrame
df_scores = benchmark.apply(lambda r: get_sims(vectors_dict, "EMLAP", r), axis=1)
scores_df = pd.DataFrame(list(df_scores))  # expand dicts into columns

In [113]:
kv = vectors_dict["EMLAP"]       # keyed vectors model
kv_vocab = set(kv.index_to_key)   # list of words in the model
emlap_vocab = set(emlap_words)

In [115]:
# 1. Intersection (words in both)
overlap = kv_vocab & emlap_vocab
print(len(emlap_vocab))
print(len(kv_vocab))
print("Overlap size:", len(overlap))

7145
7145
Overlap size: 7145


In [106]:
len(scores_df)

2759

In [107]:
scores_df

,target_present,sim_related,sim_unrelated1,sim_unrelated2,sim_unrelated3
0,False,NaN,NaN,NaN,NaN
1,False,NaN,NaN,NaN,NaN
2,True,0.163185,0.254781,NaN,NaN
3,True,0.220429,0.194320,0.381667,NaN
4,False,NaN,NaN,NaN,NaN
...,...,...,...,...,...
2754,False,NaN,NaN,NaN,NaN
2755,True,0.263835,NaN,NaN,NaN
2756,True,NaN,0.174714,NaN,NaN
2757,True,0.568247,0.108350,NaN,0.117863


In [108]:
for simcol in ["sim_related", "sim_unrelated1", "sim_unrelated2", "sim_unrelated3"]:
    print(sum(scores_df[simcol]<0))

3
19
14
4


In [109]:
# Identify sim_* columns
sim_cols = ["sim_unrelated1", "sim_unrelated2", "sim_unrelated3"]

# Filter: target_present == True and all sim_* not NaN
filtered = scores_df[
    (scores_df["target_present"] == True) &
    (scores_df["sim_related"].notnull()) &
    (scores_df[sim_cols].notna().sum(axis=1)>0)
]

len(filtered)

850

In [110]:
filtered

,target_present,sim_related,sim_unrelated1,sim_unrelated2,sim_unrelated3
2,True,0.163185,0.254781,NaN,NaN
3,True,0.220429,0.194320,0.381667,NaN
5,True,0.352966,0.267360,NaN,NaN
8,True,0.190818,0.096643,0.278960,0.194063
12,True,0.477132,0.193057,NaN,NaN
...,...,...,...,...,...
2744,True,0.710797,0.184392,NaN,NaN
2748,True,0.489439,0.220101,0.181135,0.320150
2751,True,0.459183,0.184579,NaN,NaN
2753,True,0.331682,0.278566,NaN,NaN


In [111]:
vectors_dict["EMLAP"].vectors.shape

(7145, 100)

In [92]:
emlap_words = filtered_vocab_df[filtered_vocab_df["EMLAP"] >= 10]["word"].tolist()

In [93]:
sum(benchmark[["unrelated1", "unrelated2", "unrelated3"]].isin(emlap_words).sum(axis=1) > 0)

2396

In [117]:
def qualifies(row, vocab):
    return (
        row["target"] in vocab
        and row["related"] in vocab
        and any(row[c] in vocab for c in ["unrelated1","unrelated2","unrelated3"])
    )

cnt = benchmark.apply(lambda r: qualifies(r, kv_vocab), axis=1).sum()
print(cnt)#%%

850


In [43]:
# List all your similarity columns
sim_cols = ["sim_related", "sim_unrelated1", "sim_unrelated2", "sim_unrelated3"]

# Create a boolean mask where sim_related equals the row-wise max of all sim_ columns
mask = filtered["sim_related"] == filtered[sim_cols].max(axis=1)

In [44]:
mask.sum() / len(filtered)

0.8388235294117647

In [45]:
len(benchmark)

2759

In [123]:
sim_cols = ["sim_related", "sim_unrelated1", "sim_unrelated2", "sim_unrelated3"]
sim_unrelated_cols = ["sim_unrelated1", "sim_unrelated2", "sim_unrelated3"]

def evaluate_submodel(vectors_dict, key, benchmark):
    df_scores = benchmark.apply(lambda r: get_sims(vectors_dict, key, r), axis=1)
    scores_df = pd.DataFrame(list(df_scores))

    out = {"submodel": key}

    # --- Coverage: target & related present & at least one unrelated present
    present_mask = (
        (scores_df["target_present"] == True) &
        (scores_df["sim_related"].notna()) &
        (scores_df[sim_unrelated_cols].notna().any(axis=1))
    )
    filtered_min1 = scores_df[present_mask]

    # Accuracy: allow ties as correct (use idxmax if you prefer strict top-1)
    if len(filtered_min1) > 0:
        row_max = filtered_min1[sim_cols].max(axis=1)
        acc_mask = np.isclose(filtered_min1["sim_related"], row_max)
        out["covered_min1"]  = len(filtered_min1) / len(benchmark)
        out["accuracy_min1"] = acc_mask.mean()
    else:
        out["covered_min1"] = 0.0
        out["accuracy_min1"] = np.nan

    # --- Coverage: all four sims are present (no NaNs)
    all_present = scores_df[sim_cols].notna().all(axis=1)
    filtered_all = scores_df[all_present]

    if len(filtered_all) > 0:
        # strict or tie-aware—pick one:
        # top_col = filtered_all[sim_cols].idxmax(axis=1); acc_mask_all = (top_col == "sim_related")
        row_max_all = filtered_all[sim_cols].max(axis=1)
        acc_mask_all = np.isclose(filtered_all["sim_related"], row_max_all)

        out["covered_all"]  = len(filtered_all) / len(benchmark)
        out["accuracy_all"] = acc_mask_all.mean()
    else:
        out["covered_all"] = 0.0
        out["accuracy_all"] = np.nan

    return out

benchmark_results = [evaluate_submodel(vectors_dict, k, benchmark) for k in vectors_dict.keys()]
benchmark_results_df = pd.DataFrame(benchmark_results) # .sort_values("submodel")
print(benchmark_results_df)

                                      submodel  covered_min1  accuracy_min1  \
0                         NOSCEMUS - 1501-1550      0.402320       0.909910   
1                         NOSCEMUS - 1551-1600      0.413556       0.908852   
2                         NOSCEMUS - 1601-1650      0.409931       0.904509   
3                         NOSCEMUS - 1651-1700      0.413556       0.899211   
4                 NOSCEMUS - Alchemy/Chemistry      0.390721       0.868275   
5   NOSCEMUS - Astronomy/Astrology/Cosmography      0.412831       0.913960   
6                           NOSCEMUS - Biology      0.413193       0.913158   
7             NOSCEMUS - Geography/Cartography      0.406307       0.912578   
8                       NOSCEMUS - Mathematics      0.403407       0.881402   
9                          NOSCEMUS - Medicine      0.415005       0.926638   
10       NOSCEMUS - Meteorology/Earth sciences      0.408119       0.927176   
11                          NOSCEMUS - Physics      

In [124]:
print(benchmark_results_df)

                                      submodel  covered_min1  accuracy_min1  \
0                         NOSCEMUS - 1501-1550      0.402320       0.909910   
1                         NOSCEMUS - 1551-1600      0.413556       0.908852   
2                         NOSCEMUS - 1601-1650      0.409931       0.904509   
3                         NOSCEMUS - 1651-1700      0.413556       0.899211   
4                 NOSCEMUS - Alchemy/Chemistry      0.390721       0.868275   
5   NOSCEMUS - Astronomy/Astrology/Cosmography      0.412831       0.913960   
6                           NOSCEMUS - Biology      0.413193       0.913158   
7             NOSCEMUS - Geography/Cartography      0.406307       0.912578   
8                       NOSCEMUS - Mathematics      0.403407       0.881402   
9                          NOSCEMUS - Medicine      0.415005       0.926638   
10       NOSCEMUS - Meteorology/Earth sciences      0.408119       0.927176   
11                          NOSCEMUS - Physics      

In [127]:
benchmark_results_df.round(2).to_csv("../data/benchmark_results.csv")